In [ ]:
# Adaptive Ensemble-Guided Hybrid Quantum-Classical Framework

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from qiskit import __version__
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_aer import Aer

RANDOM_STATE = 42
ALPHA = 0.6
BETA = 0.4
THRESHOLD_VALUES = [0.02,0.04,0.06,0.08,0.10,0.12,0.14,0.16,0.18,0.20]

print(f"Qiskit version: {__version__}")

In [ ]:
backend = Aer.get_backend("aer_simulator")
print("Using backend:", backend.name)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

RANDOM_STATE = 42

# =========================
# 1. Load inbuilt dataset
# =========================
breast = load_breast_cancer()

X = breast.data
y = breast.target

breast_df = pd.DataFrame(X, columns=breast.feature_names)
breast_df["diagnosis"] = y

print("Original Dataset Shape:", breast_df.shape)

# =========================
# 2. Train-test split
# =========================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# =========================
# 3. Standard Scaling
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# =========================
# 4. PCA: 30 -> 4
# =========================
pca = PCA(n_components=4, random_state=RANDOM_STATE)
X_train = pca.fit_transform(X_train_scaled)
X_test = pca.transform(X_test_scaled)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_,4))
print("Total retained variance:", np.sum(pca.explained_variance_ratio_))

In [ ]:
# 2) Classical ensemble training
rf_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
lr_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
gb_model = GradientBoostingClassifier(random_state=RANDOM_STATE)

rf_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
lr_pred = lr_model.predict(X_test)
gb_pred = gb_model.predict(X_test)

rf_conf = rf_model.predict_proba(X_test).max(axis=1)
lr_conf = lr_model.predict_proba(X_test).max(axis=1)
gb_conf = gb_model.predict_proba(X_test).max(axis=1)

rf_acc = accuracy_score(y_test, rf_pred)
lr_acc = accuracy_score(y_test, lr_pred)

gb_acc = accuracy_score(y_test, gb_pred)

In [ ]:
# 3) Ensemble consensus prediction + ambiguity score engine

def majority_vote(pred_a, pred_b, pred_c):
    stacked = np.vstack([pred_a, pred_b, pred_c])
    # For binary classification, majority vote is equivalent to >=2 positives.
    return (stacked.sum(axis=0) >= 2).astype(int)

ensemble_pred = majority_vote(rf_pred, lr_pred, gb_pred)

average_confidence = np.mean(np.vstack([rf_conf, lr_conf, gb_conf]), axis=0)
confidence_variance = np.var(np.vstack([rf_conf, lr_conf, gb_conf]), axis=0)
ambiguity_score_test = ALPHA * (1 - average_confidence) + BETA * confidence_variance

print("Ambiguity score summary:")
print(pd.Series(ambiguity_score_test).describe())

In [ ]:
# 4) Adaptive threshold sweep with VQC routing
# Compute ambiguity scores for TRAIN split as routing source for VQC training.
rf_pred_train = rf_model.predict(X_train)
lr_pred_train = lr_model.predict(X_train)
gb_pred_train = gb_model.predict(X_train)

rf_conf_train = rf_model.predict_proba(X_train).max(axis=1)
lr_conf_train = lr_model.predict_proba(X_train).max(axis=1)
gb_conf_train = gb_model.predict_proba(X_train).max(axis=1)

avg_conf_train = np.mean(np.vstack([rf_conf_train, lr_conf_train, gb_conf_train]), axis=0)
var_conf_train = np.var(np.vstack([rf_conf_train, lr_conf_train, gb_conf_train]), axis=0)
ambiguity_score_train = ALPHA * (1 - avg_conf_train) + BETA * var_conf_train

results = []
best_hybrid_pred = None
best_threshold = None
best_hybrid_acc = -1

for threshold in THRESHOLD_VALUES:
    amb_train_idx = np.where(ambiguity_score_train > threshold)[0]
    amb_test_idx = np.where(ambiguity_score_test > threshold)[0]

    hybrid_pred = ensemble_pred.copy()

    # Train VQC only when ambiguous train/test subsets are viable.
    if len(amb_train_idx) > 1 and len(np.unique(y_train[amb_train_idx])) > 1 and len(amb_test_idx) > 0:
        feature_map = ZZFeatureMap(feature_dimension=4, reps=1)
        ansatz = RealAmplitudes(num_qubits=4, reps=1)
        optimizer = COBYLA(maxiter=100)

        vqc = VQC(
            feature_map=feature_map,
            ansatz=ansatz,
            loss="cross_entropy",
            optimizer=optimizer,
            quantum_instance=backend,
        )

        vqc.fit(X_train[amb_train_idx], y_train[amb_train_idx])
        hybrid_pred[amb_test_idx] = vqc.predict(X_test[amb_test_idx])

    # Metrics for this threshold
    hybrid_acc = accuracy_score(y_test, hybrid_pred)
    precision = precision_score(y_test, hybrid_pred, zero_division=0)
    recall = recall_score(y_test, hybrid_pred, zero_division=0)
    f1 = f1_score(y_test, hybrid_pred, zero_division=0)
    malignant_recall = recall_score(y_test, hybrid_pred, pos_label=1, zero_division=0)

    results.append(
        {
            "threshold": round(float(threshold), 2),
            "hybrid_accuracy": hybrid_acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "malignant_recall": malignant_recall,
            "quantum_samples_used": int(len(amb_test_idx)),
        }
    )

    if hybrid_acc > best_hybrid_acc:
        best_hybrid_acc = hybrid_acc
        best_threshold = float(threshold)
        best_hybrid_pred = hybrid_pred.copy()

results_df = pd.DataFrame(results)
results_df = results_df[results_df["quantum_samples_used"] > 0].copy()

results_df = results_df.sort_values("threshold").reset_index(drop=True)
results_df

In [ ]:
# 5) Final results table and best-threshold selection
if len(results_df) == 0:
    raise ValueError("No adaptive threshold rows with quantum routing were available.")

best_row = results_df.loc[results_df["hybrid_accuracy"].idxmax()].copy()

print("Best threshold by hybrid accuracy:", best_row["threshold"])
print("Best hybrid accuracy:", round(best_row["hybrid_accuracy"], 4))
print("\nAdaptive threshold experiment results (quantum routed only):")
display(results_df)

comparison_df = pd.DataFrame(
    {
        "Model": [
            "Random Forest",
            "Logistic Regression",
            "Gradient Boosting",
            "Proposed Adaptive Hybrid Quantum Model",
        ],
        "Accuracy": [
            rf_acc,
            lr_acc,
            gb_acc,
            best_row["hybrid_accuracy"],
        ],
    }
)

display(
    comparison_df.style.format({"Accuracy": "{:.4f}"}).set_caption("Model Accuracy Comparison")
)


In [ ]:
# A) Ambiguity score distribution histogram
plt.figure(figsize=(8, 5))
plt.hist(ambiguity_score_test, bins=20, color="steelblue", alpha=0.8, edgecolor="black")
plt.axvline(best_threshold, color="crimson", linestyle="--", linewidth=2, label=f"Best threshold = {best_threshold:.2f}")
plt.title("Ambiguity Score Distribution (Test Set)")
plt.xlabel("Ambiguity Score")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# B) Threshold vs Hybrid Accuracy
plt.figure(figsize=(8, 5))
plt.plot(results_df["threshold"], results_df["hybrid_accuracy"], marker="o", linewidth=2)
plt.title("Threshold vs Hybrid Accuracy")
plt.xlabel("Ambiguity Threshold")
plt.ylabel("Hybrid Accuracy")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# C) Threshold vs Number of Quantum Routed Samples
plt.figure(figsize=(8, 5))
plt.plot(results_df["threshold"], results_df["quantum_samples_used"], marker="s", color="darkorange", linewidth=2)
plt.title("Threshold vs Number of Quantum Routed Samples")
plt.xlabel("Ambiguity Threshold")
plt.ylabel("Quantum-Routed Test Samples")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# D) Comparison bar chart for individual classical models and the best adaptive hybrid
comparison_labels = [
    "Random Forest Accuracy",
    "Logistic Regression Accuracy",
    "Gradient Boosting Accuracy",
    "Best Adaptive Hybrid Accuracy",
]
comparison_scores = [rf_acc, lr_acc, gb_acc, best_hybrid_acc]

plt.figure(figsize=(9, 5))
bars = plt.bar(
    comparison_labels,
    comparison_scores,
    color=["#4C78A8", "#F58518", "#54A24B", "#3E9651"],
)
plt.ylim(0, 1.0)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}", ha="center")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# E) Confusion matrix for best hybrid model
cm_best = confusion_matrix(y_test, best_hybrid_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=["Benign", "Malignant"])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Best Hybrid Confusion Matrix (threshold={best_threshold:.2f})")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Markdown, display

best_metrics = results_df.loc[results_df["threshold"] == round(best_threshold, 2)].iloc[0]

summary_md = f"""## Best Hybrid Model Performance

- **Optimal threshold:** {best_metrics['threshold']:.2f}
- **Hybrid accuracy:** {best_metrics['hybrid_accuracy']:.4f}
- **Precision:** {best_metrics['precision']:.4f}
- **Recall:** {best_metrics['recall']:.4f}
- **F1:** {best_metrics['f1']:.4f}
- **Malignant recall:** {best_metrics['malignant_recall']:.4f}
- **Quantum samples used:** {int(best_metrics['quantum_samples_used'])}
"""

display(Markdown(summary_md))


## Adaptive Hybrid Experiment Notes

- Classical stage uses a three-model ensemble (RF, Logistic Regression, Gradient Boosting).
- Ambiguity score is computed as: `0.6*(1-average_confidence) + 0.4*(confidence_variance)`.
- Quantum routing is adaptive across thresholds `0.02` to `0.20`.
- VQC core is preserved with `ZZFeatureMap(4, reps=1)`, `RealAmplitudes(4, reps=1)`, and `COBYLA(maxiter=100)`.


In [ ]:
# Per-sample test diagnostics (requested prediction/confidence fields)
test_diagnostics_df = pd.DataFrame(
    {
        "y_true": y_test,
        "rf_pred": rf_pred,
        "rf_conf": rf_conf,
        "lr_pred": lr_pred,
        "lr_conf": lr_conf,
        "gb_pred": gb_pred,
        "gb_conf": gb_conf,
        "average_confidence": average_confidence,
        "confidence_variance": confidence_variance,
        "ambiguity_score": ambiguity_score_test,
    }
)

display(test_diagnostics_df.head(10))


## Reproducibility

- `random_state=42` is fixed for train/test split and all classical models.
- PCA transformation is fit on train and applied to test.
- Threshold sweep is deterministic for the same environment and package versions.

## End of Adaptive Ensemble-Guided Hybrid Workflow